In [ ]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')

# Modelling and Forecasting
# ==============================================================================
import xgboost
import skforecast
import sklearn
from xgboost import XGBRegressor
from sklearn.feature_selection import RFECV
from skforecast.ForecasterAutoreg import ForecasterAutoreg
from skforecast.model_selection import bayesian_search_forecaster
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import select_features
import shap

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

color = '\033[1m\033[38;5;208m'
print(f"{color}Version skforecast: {skforecast.__version__}")
print(f"{color}Version scikit-learn: {sklearn.__version__}")
print(f"{color}Version xgboost: {xgboost.__version__}")
print(f"{color}Version pandas: {pd.__version__}")
print(f"{color}Version numpy: {np.__version__}")

In [ ]:
df_import = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/Aggregated_data_20241019.parquet', engine='pyarrow')

df = df_import

store_numbers = df['store_nbr'].nunique()
item_numbers = df['item_nbr'].nunique()
print(store_numbers)
print(item_numbers)

In [ ]:
filtered_df = df[df['item_family'].isin(['GROCERY I', 'BREAD/BAKERY', 'SEAFOOD'])]

# Step 1: Identify item_nbr values that have >0 unit_sales across all store_nbr
# Group by item_nbr and store_nbr, checking if all unit_sales are >0 for each item_nbr across stores
items_all_positive_sales = (
    df[df['unit_sales'] > 0]
    .groupby('item_nbr')['store_nbr']
    .nunique()
)

# Keep only item_nbrs that appear in all store_nbr (assuming we know the total store count, e.g., 54)
total_stores = df['store_nbr'].nunique()
items_all_positive_sales = items_all_positive_sales[items_all_positive_sales == total_stores].index

# Step 2: Filter the original DataFrame to include only these item_nbr values
filtered_df = df[df['item_nbr'].isin(items_all_positive_sales)]

# Display the filtered data
filtered_df.head()

df = filtered_df

store_numbers = df['store_nbr'].unique().tolist()
item_numbers = df['item_nbr'].unique().tolist()
print(store_numbers)
print(item_numbers)
n_store_numbers = df['store_nbr'].nunique()
n_item_numbers = df['item_nbr'].nunique()
print(store_numbers)
print(item_numbers)

# Select the first 5 numbers from each list
first_5_stores = store_numbers[:5]
first_5_items = item_numbers[:5]

print("First 5 unique store numbers:", first_5_stores)
print("First 5 unique item numbers:", first_5_items)

In [ ]:
# Assuming df is your DataFrame

# Define the first 5 unique store and item numbers
first_5_stores = [1, 2, 3, 4, 5]
first_5_items = [103520, 105693, 105857, 106716, 108079]

# Loop over each combination of store and item
for store in first_5_stores:
    for item in first_5_items:
        # Filter the DataFrame for the current store and item
        data = df[(df['store_nbr'] == store) & (df['item_nbr'] == item)]
        data.reset_index(drop=True, inplace=True)

        # Check if data is empty
        if data.empty:
            print(f"No data for store {store} and item {item}. Skipping...")
            continue
        
        # Define training and test periods
        start_train = 0
        end_train = 189
        end_test = 215
        
        # Split the data
        data_train = data.loc[start_train:end_train, :]
        data_test = data.loc[end_train:end_test, :]
        data_val = data.loc[end_test:, :]
        
        # Define the lags you want to include (excluding lag 1)
        lags_to_include = list(range(2, 53))  # This creates a list of lags from 2 to 52

        # Create forecaster with specified lags
        forecaster = ForecasterAutoreg(
            regressor=XGBRegressor(random_state=15926, enable_categorical=True),
            lags=lags_to_include  # Pass the updated list of lags
        )

        # Train forecaster
        forecaster.fit(y=data.loc[:end_train, 'unit_sales'])

        # Perform predictions
        predictions = forecaster.predict(steps=2)

        # Perform backtesting with MAPE
        metric, backtest_predictions = backtesting_forecaster(
            forecaster=forecaster,
            y=data['unit_sales'],
            steps=2,
            metric='mean_absolute_percentage_error',  # Change metric to MAPE
            initial_train_size=len(data[:end_train]),
            refit=False,
            n_jobs='auto',
            verbose=False,  # Change to False to see less information
            show_progress=True
        )

        # Display predictions
        print(f"Predictions for store {store}, item {item}:\n", metric:.2f)
        print("-----------------------------------------------------\n")


In [ ]:
# Step 1: Filter the DataFrame
data = df[(df['store_nbr'] == 1) & (df['item_nbr'] == 103520)]

data.reset_index(drop=True, inplace=True)
data

In [245]:
# # Step 1: Filter the DataFrame
# filtered_df = df[(df['store_nbr'] == 1) & (df['item_family'] == 'GROCERY I')]

# # Step 2: Group by week_number_cum and sum the unit_sales
# weekly_sales = filtered_df.groupby('date')['unit_sales'].sum().reset_index()

# # Step 3: Rename columns for clarity (optional)
# weekly_sales.columns = ['date', 'unit_sales']

# # Display the new DataFrame
# weekly_sales

# data = weekly_sales

# data



In [ ]:
start_train = 0
end_train = 189
end_test = 215
data_train = data.loc[start_train:end_train, :]
data_test   = data.loc[end_train:end_test, :]
data_val  = data.loc[end_test:, :]

print(f"Weeks train: {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Weeks test: {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_val)})")
print(f"Weeks validate: {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_test)})")

# df_train = df[(df['week_number_cum'] > 138) & (df['week_number_cum'] <= 190)]
# df_test = df[(df['week_number_cum'] > 190) & (df['week_number_cum'] <= 216)]
# df_validate = df[df['week_number_cum'] > 216]

In [ ]:
# Interactive plot of time series
# ==============================================================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=data_train.index, y=data_train['unit_sales'], mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=data_test.index, y=data_test['unit_sales'], mode='lines', name='Test'))
fig.add_trace(go.Scatter(x=data_val.index, y=data_val['unit_sales'], mode='lines', name='Validation'))
fig.update_layout(
    title  = 'Number of unit_sales',
    xaxis_title="Time",
    yaxis_title="Unit_sales",
    legend_title="Partition:",
    width=800,
    height=350,
    margin=dict(l=20, r=20, t=35, b=20),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1,
        xanchor="left",
        x=0.001
    )
)
#fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 2))
plot_acf(data['unit_sales'], ax=ax, lags=52)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))
plot_pacf(data['unit_sales'], ax=ax, lags=52, method='ywm')
plt.show()

In [ ]:
# Define the lags you want to include (excluding lag 1)
lags_to_include = list(range(2, 53))  # This creates a list of lags from 2 to 52

# Create forecaster with specified lags
forecaster = ForecasterAutoreg(
                 regressor = XGBRegressor(random_state=15926, enable_categorical=True),
                 lags      = lags_to_include  # Pass the updated list of lags
             )

# Train forecaster
# ==============================================================================

# Fit the forecaster with the training data up to end_train
forecaster.fit(y=data.loc[:end_train, 'unit_sales'])

# Display the forecaster
forecaster

In [ ]:
forecaster.predict(steps=2)

In [ ]:
# Perform backtesting with MAPE
metric, predictions = backtesting_forecaster(
    forecaster         = forecaster,
    y                  = data['unit_sales'],
    steps              = 2,
    metric             = 'mean_absolute_percentage_error',  # Change metric to MAPE
    initial_train_size = len(data[:end_train]),
    refit              = False,
    n_jobs             = 'auto',
    verbose            = False,  # Change to False to see less information
    show_progress      = True
)

# Display predictions
predictions.head()

In [ ]:
metric

In [254]:
data_test_pred = data_test

In [ ]:
data_test_pred['pred'] = predictions['pred']

In [ ]:
data_test_pred

In [ ]:
data_test_pred['bias'] = data_test_pred['unit_sales'] - data_test_pred['pred']
data_test_pred['accuracy'] = (1- (np.abs(data_test_pred['bias']) / data_test_pred['unit_sales'])) *100


In [ ]:
bias = np.mean(data_test_pred['bias'])
sd_bias = np.std(data_test_pred['bias'])
#accuracy = (1 - (np.mean(np.abs(data_test_pred['bias'])) / np.mean(data_test_pred['unit_sales']))) * 100
accuracy = np.mean(data_test_pred['accuracy'])
sd_accuracy = np.std(data_test_pred['accuracy'])

print(f'Bias: {bias:.2f} with SD {sd_bias:.2f}')
print(f'Accuracy: {accuracy:.2f}% with SD {sd_accuracy:.2f}')

In [ ]:
# Ensure 'date' column is in datetime format
data_test_pred['date'] = pd.to_datetime(data_test_pred['date'])

# Set date as index (optional, useful for time series plotting)
data_test_pred.set_index('date', inplace=True)

# Plotting actual vs. predicted values
plt.figure(figsize=(14, 7))
plt.plot(data_test_pred.index, data_test_pred['unit_sales'], label='Actual Unit Sales', color='blue', marker='o')
plt.plot(data_test_pred.index, data_test_pred['pred'], label='Predicted Unit Sales', color='orange', marker='x')

# Adding titles and labels
plt.title('Actual vs. Predicted Unit Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Unit Sales')
plt.legend()
plt.grid(True)

# Show plot
plt.show()
